# Türkiye Odaklı Maaşla Yatırım Araştırma Defteri

Bu notebook tek bir soruyu yanıtlamaya çalışır:

> Eğer son 6 ay boyunca her ay maaşımdan düzenli yatırım yapsaydım, farklı stratejilerle sonuç ne olurdu?

Bu sürüm özellikle **Türkiye piyasasını anlamayı kolaylaştırmak** için sade tutuldu.

- Çıktılar yalnızca notebook içinde gösterilir
- Dosya kaydı / dump yapılmaz
- Grafikler Plotly ile üretilir
- Para eksenleri ve önemli metrikler **Türk lirası (₺)** biçiminde gösterilir
- LLM bu sürümde **kapalıdır**

Stratejiler:
- **Eşit Ağırlık**
- **HRP (Hierarchical Risk Parity)**
- **Black-Litterman (kurallı view)**


In [ ]:
# Colab kurulum hücresi
!pip -q install yfinance PyPortfolioOpt plotly pandas numpy scipy scikit-learn


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
import numpy as np
import pandas as pd
import yfinance as yf

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pypfopt.expected_returns import mean_historical_return, returns_from_prices
from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.hierarchical_portfolio import HRPOpt
from pypfopt.black_litterman import BlackLittermanModel, market_implied_risk_aversion
from pypfopt.efficient_frontier import EfficientFrontier


In [ ]:
CONFIG = {
    'tickers': ['XU100.IS', 'ZGOLD.IS', 'USDTRY=X'],
    'ticker_labels': {
        'XU100.IS': 'BIST 100',
        'ZGOLD.IS': 'Altın BYF',
        'USDTRY=X': 'Dolar/TL'
    },
    'salary_amount_try': 25000,
    'lookback_days': 252 * 2,
    'test_months': 6,
    'download_period': '3y',
    'risk_free_rate': 0.0,
    'max_weight': 0.75,
    'gold_ticker': 'ZGOLD.IS',
    'gold_min': 0.10,
    'gold_max': 0.60,
}

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

CONFIG

In [ ]:
def try_fmt(x):
    if pd.isna(x):
        return '-'
    return f'₺{x:,.0f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

def pct_fmt(x):
    if pd.isna(x):
        return '-'
    return f'{x*100:.2f}%'

def style_plotly(fig, title=None):
    fig.update_layout(
        template='plotly_white',
        title=title,
        legend_title_text='',
        hovermode='x unified',
        margin=dict(l=40, r=40, t=60, b=40),
        height=500
    )
    return fig

def set_try_axis(fig, axis='y'):
    if axis == 'y':
        fig.update_yaxes(tickprefix='₺', separatethousands=True)
    elif axis == 'x':
        fig.update_xaxes(tickprefix='₺', separatethousands=True)
    return fig

def relabel_cols(df):
    return df.rename(columns=CONFIG['ticker_labels'])

def clean_weights(weights_dict, tickers):
    weights = pd.Series(weights_dict).reindex(tickers).fillna(0.0)
    if weights.sum() <= 0:
        weights[:] = 1 / len(weights)
    weights = weights / weights.sum()
    return weights

def apply_gold_band(weights, gold_ticker, gold_min, gold_max):
    w = weights.copy().astype(float)
    if gold_ticker not in w.index:
        return w / w.sum()
    gold_w = w[gold_ticker]
    target_gold = min(max(gold_w, gold_min), gold_max)
    delta = target_gold - gold_w
    if abs(delta) < 1e-12:
        return w / w.sum()
    others = w.drop(gold_ticker)
    if others.sum() <= 0:
        w[:] = 0
        w[gold_ticker] = 1.0
        return w
    others = others / others.sum()
    w[gold_ticker] = target_gold
    w.loc[others.index] = (1 - target_gold) * others
    return w / w.sum()

def get_rule_based_views(price_window):
    rets = price_window.pct_change().dropna()
    ann_ret = rets.mean() * 252
    vol = rets.std() * np.sqrt(252)
    score = ann_ret / vol.replace(0, np.nan)
    score = score.replace([np.inf, -np.inf], np.nan).fillna(0)
    views = score.clip(-0.15, 0.15)
    confidences = (score.abs() / (score.abs().max() + 1e-9)).clip(0.2, 0.8)
    return views.to_dict(), confidences.to_dict()


In [ ]:
raw = yf.download(CONFIG['tickers'], period=CONFIG['download_period'], auto_adjust=True, progress=False)

if isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Close'].copy()
else:
    prices = raw.copy()

prices = prices.dropna(how='all').ffill().dropna()
prices = prices[CONFIG['tickers']]
prices.columns = [CONFIG['ticker_labels'][c] for c in prices.columns]

assert len(prices) > CONFIG['lookback_days'] + 40, 'Yeterli veri yok.'

display(prices.tail())


## Son 2 Yıl - Normalize Fiyatlar

Burada bütün varlıklar aynı başlangıç değerine çekiliyor. Böylece **hangi varlık daha iyi performans göstermiş** daha kolay görülüyor.


In [ ]:
norm_prices = prices / prices.iloc[0] * 100
fig = px.line(norm_prices, x=norm_prices.index, y=norm_prices.columns, labels={'value': 'Başlangıç=100', 'index': 'Tarih'})
fig = style_plotly(fig, 'Normalize Fiyat Karşılaştırması')
fig.show()


## Korelasyon Isı Haritası

Bu tablo varlıkların birbirine ne kadar benzer hareket ettiğini gösterir.


In [ ]:
rets = prices.pct_change().dropna()
corr = rets.corr()
fig = px.imshow(corr, text_auto=True, aspect='auto', color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig = style_plotly(fig, 'Getiri Korelasyonları')
fig.show()


## Pseudo-Live Maaşla Yatırım Testi

Her ay başında maaştan sabit tutar ekleniyor. O tarihte gelecek bilinmiyormuş gibi yalnızca geçmiş veriye bakılıyor.


In [ ]:
def get_equal_weight(tickers):
    w = pd.Series(1 / len(tickers), index=tickers)
    return w

def get_hrp_weights(price_window):
    ret_window = price_window.pct_change().dropna()
    hrp = HRPOpt(ret_window)
    weights = hrp.optimize()
    return clean_weights(weights, price_window.columns)

def get_bl_weights(price_window):
    ret_window = returns_from_prices(price_window)
    S = CovarianceShrinkage(price_window).ledoit_wolf()
    views, confidences = get_rule_based_views(price_window)
    delta = market_implied_risk_aversion(price_window.mean(axis=1).dropna())
    market_caps = pd.Series(1.0, index=price_window.columns)
    bl = BlackLittermanModel(
        S,
        pi='equal',
        absolute_views=views,
        omega='default'
    )
    rets = bl.bl_returns()
    ef = EfficientFrontier(rets, S, weight_bounds=(0, CONFIG['max_weight']))
    ef.max_sharpe(risk_free_rate=CONFIG['risk_free_rate'])
    weights = ef.clean_weights()
    weights = clean_weights(weights, price_window.columns)
    weights = apply_gold_band(weights, CONFIG['ticker_labels'][CONFIG['gold_ticker']], CONFIG['gold_min'], CONFIG['gold_max'])
    diagnostics = pd.DataFrame({
        'view': pd.Series(views),
        'confidence': pd.Series(confidences)
    })
    return weights, diagnostics

def run_salary_backtest(prices, strategy_name):
    monthly_prices = prices.resample('MS').first().dropna()
    start_idx = len(monthly_prices) - CONFIG['test_months']
    assert start_idx > 0, 'Test için yeterli aylık veri yok.'

    capital = 0.0
    shares = pd.Series(0.0, index=prices.columns)
    history = []
    weight_history = []
    bl_diag = []

    rebalance_dates = monthly_prices.index[start_idx:]

    for i, dt in enumerate(rebalance_dates):
        capital += CONFIG['salary_amount_try']
        price_window = prices.loc[:dt].tail(CONFIG['lookback_days'])

        if strategy_name == 'Eşit Ağırlık':
            weights = get_equal_weight(prices.columns)
            diagnostics = None
        elif strategy_name == 'HRP':
            weights = get_hrp_weights(price_window)
            diagnostics = None
        elif strategy_name == 'Black-Litterman':
            weights, diagnostics = get_bl_weights(price_window)
        else:
            raise ValueError('Bilinmeyen strateji')

        current_prices = prices.loc[dt]
        portfolio_value = (shares * current_prices).sum()
        total_value = portfolio_value + capital

        target_value = weights * total_value
        shares = target_value / current_prices
        capital = 0.0

        next_dt = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        path = prices.loc[(prices.index >= dt) & (prices.index <= next_dt)]
        values = path.mul(shares, axis=1).sum(axis=1)

        for t, v in values.items():
            history.append({'date': t, 'strategy': strategy_name, 'portfolio_value': v})

        row = {'date': dt, 'strategy': strategy_name, 'capital_after_contribution': total_value}
        for c, w in weights.items():
            row[f'weight_{c}'] = w
        weight_history.append(row)

        if diagnostics is not None:
            temp = diagnostics.copy()
            temp['date'] = dt
            temp['strategy'] = strategy_name
            bl_diag.append(temp.reset_index().rename(columns={'index': 'asset'}))

    hist_df = pd.DataFrame(history).drop_duplicates(['date', 'strategy'])
    weights_df = pd.DataFrame(weight_history)
    bl_diag_df = pd.concat(bl_diag, ignore_index=True) if bl_diag else pd.DataFrame()
    return hist_df, weights_df, bl_diag_df


In [ ]:
strategies = ['Eşit Ağırlık', 'HRP', 'Black-Litterman']
all_hist = []
all_weights = []
all_bl_diag = []

for s in strategies:
    hist_df, weights_df, bl_diag_df = run_salary_backtest(prices, s)
    all_hist.append(hist_df)
    all_weights.append(weights_df)
    if not bl_diag_df.empty:
        all_bl_diag.append(bl_diag_df)

hist_df = pd.concat(all_hist, ignore_index=True)
weights_df = pd.concat(all_weights, ignore_index=True)
bl_diag_df = pd.concat(all_bl_diag, ignore_index=True) if all_bl_diag else pd.DataFrame()

display(hist_df.tail())


In [ ]:
summary_rows = []
contributed = CONFIG['salary_amount_try'] * CONFIG['test_months']

for s in strategies:
    sub = hist_df[hist_df['strategy'] == s].sort_values('date').copy()
    sub['ret'] = sub['portfolio_value'].pct_change().fillna(0)
    ending_value = sub['portfolio_value'].iloc[-1]
    net_profit = ending_value - contributed
    ann_vol = sub['ret'].std() * np.sqrt(252)
    sharpe = (sub['ret'].mean() / (sub['ret'].std() + 1e-9)) * np.sqrt(252)
    cummax = sub['portfolio_value'].cummax()
    drawdown = sub['portfolio_value'] / cummax - 1
    maxdd = drawdown.min()
    twr = ending_value / contributed - 1
    summary_rows.append({
        'Strateji': s,
        'Toplam Yatırılan': contributed,
        'Bitiş Değeri': ending_value,
        'Net Kar': net_profit,
        'TWR': twr,
        'Yıllık Volatilite': ann_vol,
        'Sharpe': sharpe,
        'Maksimum Düşüş': maxdd
    })

summary = pd.DataFrame(summary_rows)
summary_display = summary.copy()
for c in ['Toplam Yatırılan', 'Bitiş Değeri', 'Net Kar']:
    summary_display[c] = summary_display[c].map(try_fmt)
for c in ['TWR', 'Yıllık Volatilite', 'Maksimum Düşüş']:
    summary_display[c] = summary_display[c].map(pct_fmt)
summary_display['Sharpe'] = summary_display['Sharpe'].map(lambda x: f'{x:.2f}')
display(summary_display)


## Portföy Değeri Grafiği


In [ ]:
fig = px.line(hist_df, x='date', y='portfolio_value', color='strategy', labels={'portfolio_value': 'Portföy Değeri', 'date': 'Tarih', 'strategy': 'Strateji'})
fig = style_plotly(fig, 'Stratejilere Göre Portföy Değeri')
fig = set_try_axis(fig, 'y')
fig.show()


## Drawdown Grafiği


In [ ]:
dd_rows = []
for s in strategies:
    sub = hist_df[hist_df['strategy'] == s].sort_values('date').copy()
    sub['drawdown'] = sub['portfolio_value'] / sub['portfolio_value'].cummax() - 1
    dd_rows.append(sub[['date', 'strategy', 'drawdown']])
dd_df = pd.concat(dd_rows, ignore_index=True)
fig = px.line(dd_df, x='date', y='drawdown', color='strategy', labels={'drawdown': 'Drawdown', 'date': 'Tarih', 'strategy': 'Strateji'})
fig = style_plotly(fig, 'Stratejilere Göre Drawdown')
fig.update_yaxes(tickformat='.0%')
fig.show()


## Son Ağırlıklar


In [ ]:
last_weights = weights_df.sort_values('date').groupby('strategy').tail(1).copy()
weight_cols = [c for c in last_weights.columns if c.startswith('weight_')]
pretty_last_weights = last_weights[['strategy'] + weight_cols].copy()
pretty_last_weights.columns = ['Strateji'] + [c.replace('weight_', '') for c in weight_cols]
for c in pretty_last_weights.columns[1:]:
    pretty_last_weights[c] = pretty_last_weights[c].map(pct_fmt)
display(pretty_last_weights.reset_index(drop=True))


## Yeniden Dengeleme Özeti


In [ ]:
rebalance_summary = weights_df.copy()
rebalance_summary['capital_after_contribution'] = rebalance_summary['capital_after_contribution'].map(try_fmt)
for c in [col for col in rebalance_summary.columns if col.startswith('weight_')]:
    rebalance_summary[c] = rebalance_summary[c].map(pct_fmt)
display(rebalance_summary.head(20))


## Black-Litterman View Tanıları

Bu bölüm sadece Black-Litterman stratejisinin hangi görüşleri oluşturduğunu gösterir.


In [ ]:
if bl_diag_df.empty:
    print('BL tanı tablosu boş.')
else:
    diag_show = bl_diag_df.copy()
    diag_show['view'] = diag_show['view'].map(pct_fmt)
    diag_show['confidence'] = diag_show['confidence'].map(lambda x: f'{x:.2f}')
    display(diag_show.head(30))
